In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


## Answer:
### 1. Define the objective: what are you trying to predict?

The objective is to predict the final SalePrice (the target variable) for each house in the test dataset, given 79 variables (features) that describe various aspects of the residential homes.

### 2. How should you frame this problem as a machine learning problem?

We are framing this problem to be a supervised regression.

Why Supervised? We are given a labeled training dataset that contains both the features and the corresponding target variable. The model learns the relationship between these features and the price from this labeled data.

Why Regression? The target variable we are trying to predict, SalePrice, is a continuous numerical value, not a discrete category.

### 3. How should performance be measured? 

We are measuring the performance using the Root Mean Squared Logarithmic Error (RMSLE).

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
# Save the training data to a local parameter
housing_df = pd.read_csv(
    "/kaggle/input/house-prices-advanced-regression-techniques/train.csv",
    index_col="Id",
)

# Print dataset features count and type
housing_df.info()

# Print the head of the training data
housing_df.head()

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
# Calculate missing data
missing = housing_df.isnull().sum()
missing_percent = (missing / len(housing_df)) * 100

# Combine into a DataFrame for easy viewing
missing_data = pd.DataFrame({'Missing Count': missing, 'Missing Percent': missing_percent})

# Filter to only show columns with missing values and sort them
missing_data = missing_data[missing_data['Missing Count'] > 0].sort_values(by='Missing Percent', ascending=False)

print("Features with missing values:")
print(missing_data)

In [ ]:
# Plot GrLivArea vs. SalePrice to visually inspect for outliers
plt.figure(figsize=(10, 6))
sns.scatterplot(x=housing_df['GrLivArea'], y=housing_df['SalePrice'])
plt.title('GrLivArea vs. SalePrice (Identifying Outliers)')
plt.xlabel('Above Grade Living Area (sq ft)')
plt.ylabel('Sale Price')
plt.show()

In [ ]:
print("Checking for useless features (e.g., low variance):\n")
print(housing_df['Utilities'].value_counts(), "\n")
housing_df['CentralAir'].value_counts()

In [ ]:
# Plot the original SalePrice distribution
plt.figure(figsize=(12, 6))
sns.histplot(housing_df['SalePrice'], kde=True, bins=50)
plt.title('Distribution of SalePrice (Original)')
plt.xlabel('Sale Price')
plt.ylabel('Frequency')
plt.show()

# Plot the log-transformed SalePrice distribution
# We use np.log1p(x) which calculates log(1 + x) to avoid errors if prices were 0
plt.figure(figsize=(12, 6))
sns.histplot(np.log1p(housing_df['SalePrice']), kde=True, bins=50)
plt.title('Log-Transformed Distribution of SalePrice')
plt.xlabel('Log(SalePrice + 1)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Box plot for OverallQual (an ordinal feature)
plt.figure(figsize=(14, 7))
sns.boxplot(x='OverallQual', y='SalePrice', data=housing_df)
plt.title('SalePrice vs. Overall Quality')
plt.show()

# Box plot for Neighborhood (a nominal feature)
plt.figure(figsize=(18, 9))
sns.boxplot(x='Neighborhood', y='SalePrice', data=housing_df)
plt.title('SalePrice vs. Neighborhood')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Calculate the correlation matrix
# We must first select only the numerical columns
corr_matrix = housing_df.select_dtypes(include=[np.number]).corr()

# Print the top 10 features most correlated with SalePrice
print("\nTop 10 features most correlated with SalePrice:")
top_corr_features = corr_matrix['SalePrice'].sort_values(ascending=False).drop('SalePrice').head(10)
print(top_corr_features)

# Plot the heatmap
# This heatmap is large and can be hard to read, but gives a good overview
plt.figure(figsize=(20, 16))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', fmt='.1f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

## Answer:
#### 1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)

The dataset contains a mix of feature types. Numerical features include continuous (`int64`/`float64`) variables like `lotArea` and `GrLivArea`. Temporal features relate to dates, like `YearBuilt` and `YrSold`. Categorical features (object type) are split into:

- Nominal: Categories with no order (e.g., `Neighborhood`, `MSZoning`).

- Ordinal: Categories with a clear ranking (e.g., `ExterQual`, `BsmtQual`, `OverallQual`) which are represented as both objects ("Gd", "TA") and integers (1-10).

#### 2. What features have missing values?

As the list shows, 19 features have missing values. The features with the most missing data (over 80%) are `PoolQC`, `MiscFeature`, `Alley`, and `Fence`. Other notable missing features include `LotFrontage` (17.7%) and many `Garage...` and `Bsmt...` features (around 2-5%).

#### 3. Are there any outliers? If so, how can you tell?

Yes. The `GrLivArea` vs. `SalePrice` scatter plot clearly shows two data points in the bottom-right corner. These houses have a very large `GrLivArea` (over 4,000 sq ft) but an unusually low `SalePrice`, detaching them from the main trend.

#### 4. What features appear usefulness for the task? Are there any that appear useless? Why?

- Useful: Features with high correlation to `SalePrice` are very useful. The top 5 are: `OverallQual` (0.79), `GrLivArea` (0.71), `GarageCars` (0.64), `GarageArea` (0.62), and `TotalBsmtSF` (0.61). The box plots for `OverallQual` and `Neighborhood` also confirm their strong relationship with price.

- Useless: The `Utilities` feature appears useless. The output shows 1,459 of 1,460 houses have the same value ("AllPub"). A feature with no variance cannot be used to predict a variable target.

#### 5. Identify and characterize the target distribution.

The distribution of the target variable, `SalePrice`, is positively skewed (or "right-skewed"). The first histogram shows a long tail of very expensive houses. The second histogram confirms that a log-transformation (`np.log1p`) makes the distribution much more symmetrical and normal (bell-shaped).

#### 6. Visualize the data.

Visualizations were successfully created. We used:

- `sns.histplot` (histogram) to analyze the target distribution.

- `sns.scatterplot` to identify outliers.

- `sns.boxplot` to observe the relationship between categorical features (`OverallQual`, `Neighborhood`) and `SalePrice`.

- `sns.heatmap` to study the correlation matrix.

#### 7. Study the correlations between attributes.

The correlation heatmap and list show a strong positive correlation between `SalePrice` and key features like `OverallQual`, `GrLivArea`, and `GarageCars`. Also, it showed Multicollinearity (features that are highly correlated with each other), which can be an issue for some models. Strong examples include `GarageCars` and `GarageArea`, `TotalBsmtSF` and `1stFlrSF`, and `GrLivArea` and `TotRmsAbvGrd`.

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
# Import Scikit Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder,
    FunctionTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from scipy.stats import uniform, randint, loguniform

In [ ]:
# Drop Outliers
housing_df = housing_df.drop(
        housing_df[(housing_df['GrLivArea'] > 4000) & (housing_df['SalePrice'] < 300_000)].index
    )

# Separating Features (ff) and target (ty)

ff = housing_df.drop('SalePrice', axis=1)
ty = np.log1p(housing_df['SalePrice'])

# Splitting the data
ff_train, ff_test, ty_train, ty_test = train_test_split(ff, ty, test_size=0.2, random_state=42)

In [ ]:
def create_age_features(df):
    """
    Engineering 'HouseAge' and 'IsRemodeled' features.
    Returns only the new features.
    """
    # Create a copy to avoid SettingWithCopyWarning
    df_new = df.copy()
    
    # Ensure dtypes are numeric
    df_new['YrSold'] = pd.to_numeric(df_new['YrSold'])
    df_new['YearBuilt'] = pd.to_numeric(df_new['YearBuilt'])
    df_new['YearRemodAdd'] = pd.to_numeric(df_new['YearRemodAdd'])

    df_new['HouseAge'] = df_new['YrSold'] - df_new['YearBuilt']
    df_new['IsRemodeled'] = (df_new['YearRemodAdd'] != df_new['YearBuilt']).astype(int)
    
    # Return only the new columns
    return df_new[['HouseAge', 'IsRemodeled']]

def convert_dtypes(df):
    """
    Converts columns to the types defined in the global numeric_features_dict.
    """
    # Filter the dictionary to only include columns present in the dataframe
    # to avoid errors during pipeline execution.
    types_to_apply = {k: v for k, v in numeric_features_dict.items() if k in df.columns}
    return df.astype(types_to_apply)

In [ ]:
# Identify Features with Low correlation to the target

train_data = ff_train.join(ty_train)
corr_matrix = train_data.select_dtypes(include=[np.number]).corr()
corr_with_target = corr_matrix['SalePrice'].abs()

CORR_THRESHOLD = 0.05 
    
low_corr_features = corr_with_target[corr_with_target < CORR_THRESHOLD].index

print(low_corr_features.tolist())

In [ ]:
# Feature Selection
numeric_features = [
    "LotFrontage", 
    "LotArea", 
    "MasVnrArea", 
    "BsmtFinSF1", 
    "BsmtFinSF2",
    "BsmtUnfSF", 
    "TotalBsmtSF", 
    "1stFlrSF", 
    "2ndFlrSF", 
    "LowQualFinSF",
    "GrLivArea", 
    "GarageArea", 
    "WoodDeckSF", 
    "OpenPorchSF", 
    "EnclosedPorch",
    "3SsnPorch", 
    "ScreenPorch", 
    "PoolArea", 
    "MiscVal",
    # Discrete counts that can be scaled
    "BsmtFullBath", 
    "BsmtHalfBath", 
    "FullBath", 
    "HalfBath",
    "BedroomAbvGr", 
    "KitchenAbvGr", 
    "Fireplaces", 
    "GarageCars"
]

# Create the dictionary with float64 as dtype
numeric_features_dict = {f: np.float64 for f in numeric_features}

# Features with a clear, defined order
ord_qual_features = [
    'ExterQual',
    'ExterCond', 
    'BsmtQual', 
    'BsmtCond',
    'HeatingQC', 
    'KitchenQual', 
    'FireplaceQu', 
    'GarageQual', 
    'GarageCond'
]
quality_order = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
ord_qual_categories = [quality_order] * len(ord_qual_features)

# Other ordinal features (impute 'None' or 'mode')
ord_other_features = [
    'BsmtExposure', 
    'BsmtFinType1', 
    'BsmtFinType2', 
    'GarageFinish',
    'LotShape', 
    'LandSlope', 
    'PavedDrive', 
    'Functional'
]
ord_other_categories = [
    ['None', 'No', 'Mn', 'Av', 'Gd'],  # BsmtExposure
    ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],  # BsmtFinType1
    ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],  # BsmtFinType2
    ['None', 'Unf', 'RFn', 'Fin'],  # GarageFinish
    ['Reg', 'IR1', 'IR2', 'IR3'],  # LotShape
    ['Gtl', 'Mod', 'Sev'],  # LandSlope
    ['N', 'P', 'Y'],  # PavedDrive
    ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']  # Functional
]

nominal_mode_features = [
    'MSSubClass',
    'MSZoning', 
    'LandContour', 
    'LotConfig', 
    'Neighborhood',
    'Condition1', 
    'Condition2', 
    'BldgType', 
    'HouseStyle', 
    'RoofStyle',
    'RoofMatl', 
    'Exterior1st', 
    'Exterior2nd', 
    'Foundation',
    'Heating', 
    'CentralAir', 
    'Electrical', 
    'SaleType', 
    'SaleCondition'
]
nominal_none_features = [
    'MasVnrType', 
    'GarageType'
]
nominal_features = nominal_mode_features + nominal_none_features
engineering_features = [
    'YearBuilt',
    'YrSold',
    'YearRemodAdd'
]

passthrough_features = [
    'OverallQual', 
    'OverallCond', 
    'TotRmsAbvGrd', 
    'MoSold'
]


In [ ]:
numeric_features = [f for f in numeric_features if f not in low_corr_features]
numeric_features_dict = {f: n for f, n in numeric_features_dict.items() if f not in low_corr_features}
passthrough_features = [f for f in passthrough_features if f not in low_corr_features]

In [ ]:
# Creating the pipeline
# Numeric Features Pipline
numeric_pipe = make_pipeline(
    FunctionTransformer(
                func=convert_dtypes,
    ),
    SimpleImputer(strategy='median'),
    StandardScaler(),
    verbose=True,
).set_output(transform="pandas")

# Ordinal Quality Features Pipline
ord_qual_pipeline = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='None'),
    OrdinalEncoder(categories=ord_qual_categories, handle_unknown='use_encoded_value', unknown_value=-1)
).set_output(transform="pandas")

# Other Ordinal Features Pipelines
ord_exposure_pipe = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='None'),
    OrdinalEncoder(categories=[ord_other_categories[0]], handle_unknown='use_encoded_value', unknown_value=-1)
).set_output(transform="pandas")

ord_fintype1_pipe = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='None'),
    OrdinalEncoder(categories=[ord_other_categories[1]], handle_unknown='use_encoded_value', unknown_value=-1)
).set_output(transform="pandas")

ord_fintype2_pipe = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='None'),
    OrdinalEncoder(categories=[ord_other_categories[2]], handle_unknown='use_encoded_value', unknown_value=-1)
).set_output(transform="pandas")

ord_garagefinish_pipe = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='None'),
    OrdinalEncoder(categories=[ord_other_categories[3]], handle_unknown='use_encoded_value', unknown_value=-1)
).set_output(transform="pandas")

ord_mode_pipe = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OrdinalEncoder(categories=ord_other_categories[4:], handle_unknown='use_encoded_value', unknown_value=-1)
).set_output(transform="pandas")

# NOMINAL features Pipelines
nominal_pipe_mode = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
).set_output(transform="pandas")

nominal_pipe_none = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='None'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
).set_output(transform="pandas")

# Feature Engineering Pipeline
engineering_pipe = make_pipeline(
    SimpleImputer(strategy='median'), # Impute years just in case
    FunctionTransformer(create_age_features),
    StandardScaler()
).set_output(transform="pandas")

In [ ]:
# Combine Pipelines

preprocessor = ColumnTransformer(
    transformers=[
        # Numerical
        ('num', numeric_pipe, numeric_features),
        
        # Ordinal
        ('ord_qual', ord_qual_pipeline, ord_qual_features),
        ('ord_exp', ord_exposure_pipe, ['BsmtExposure']),
        ('ord_fin1', ord_fintype1_pipe, ['BsmtFinType1']),
        ('ord_fin2', ord_fintype2_pipe, ['BsmtFinType2']),
        ('ord_gar', ord_garagefinish_pipe, ['GarageFinish']),
        ('ord_mode', ord_mode_pipe, ['LotShape', 'LandSlope', 'PavedDrive', 'Functional']),
        
        # Nominal
        ('nom_mode', nominal_pipe_mode, nominal_mode_features),
        ('nom_none', nominal_pipe_none, nominal_none_features),
        
        # Engineered
        ('eng', engineering_pipe, engineering_features),
        
        # Passthrough
        ('pass', 'passthrough', passthrough_features)
    ],
    remainder='drop'  # Drop any columns we didn't explicitly select
).set_output(transform="pandas")
preprocessor

# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
def analyze_features(model, columns, top_n=10):
    """
    Analyzes and plots the most significant features for a model.
    Handles both 'coef_' (linear models) and 'feature_importances_' (tree models).
    """
    if hasattr(model, 'coef_'):
        importances = model.coef_
        title = 'Feature Coefficients'
    elif hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        title = 'Feature Importances'
    else:
        print(f"Model {type(model).__name__} has no feature importance attribute.")
        return

    # Create a DataFrame
    feat_df = pd.DataFrame({
        'feature': columns,
        'importance': importances
    })
    
    # Get top N features by absolute importance
    feat_df['abs_importance'] = feat_df['importance'].abs()
    top_features = feat_df.sort_values(by='abs_importance', ascending=False).head(top_n)
    
    # Plot
    print(f"\nAnalyzing features for: {type(model).__name__}")
    plt.figure(figsize=(10, top_n / 2))
    sns.barplot(x='importance', y='feature', data=top_features)
    plt.title(f'Top {top_n} {title} for {type(model).__name__}')
    plt.show()
    return top_features

def analyze_errors(y_true, y_pred, model_name):
    """
    Plots a Predicted vs. Actual scatter plot to analyze errors.
    """
    print(f"\nAnalyzing errors for: {model_name}")
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
    
    # Add a y=x line for reference
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
    
    plt.title(f'Error Analysis: Actual vs. Predicted\n({model_name})')
    plt.xlabel('Actual Log(SalePrice)')
    plt.ylabel('Predicted Log(SalePrice)')
    plt.axis('equal')
    plt.show()

In [ ]:
preprocessor.fit(ff_train, ty_train)
X_train_processed = preprocessor.transform(ff_train)
X_test_processed = preprocessor.transform(ff_test)
feature_names = X_train_processed.columns
feature_names

In [ ]:
# Model 1: Unregularized Linear Model
lin_reg = LinearRegression()
lin_reg.fit(X_train_processed, ty_train)

In [ ]:
# Model 2: Linear Model (Ridge, default alpha=1.0)
# Using Ridge(alpha=1.0) as the "default regularized" model
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X_train_processed, ty_train)

In [ ]:
# Model 3: Decision Tree (default parameters)
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(X_train_processed, ty_train)

In [ ]:
models = {
    "LinearRegression": lin_reg,
    "RidgeRegression (alpha=1.0)": ridge_reg,
    "DecisionTree": tree_reg
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test_processed)
    
    # We calculate RMSE on the log-transformed prices.

    rmse = np.sqrt(mean_squared_error(ty_test, y_pred))
    r2 = r2_score(ty_test, y_pred)
    
    results.append({
        "Model": name,
        "Test R-squared": r2,
        "Test RMSE (RMSLE)": rmse
    })

performance_df = pd.DataFrame(results).set_index("Model")
performance_df.sort_values(by="Test R-squared", ascending=False)

In [ ]:
for name, model in models.items():
    analyze_features(model, feature_names, top_n=10)

for name, model in models.items():
    y_pred = model.predict(X_test_processed)
    analyze_errors(ty_test, y_pred, name)

### Key Insights from Error Analysis
Based on the "Actual vs. Predicted" plots:

1.  **LinearRegression (Unregularized):**
    * **Result:** Complete failure. The plot shows nonsensical predictions, with a y-axis scale in the trillions.
    * **Why:** This is a classic case of **extreme overfitting**. With over 200 features (after one-hot encoding), the model found patterns in the training data noise and assigned massive coefficients, which failed completely on the test set. This highlights the necessity of regularization.

2.  **DecisionTree (Default Parameters):**
    * **Result:** Significantly better, but still flawed (R-squared ~0.77).
    * **Why:** The error plot shows two clear problems:
        * **Higher Variance:** The points form a "looser" cloud than the Ridge model, indicating predictions are less precise.
        * **Systematic Bias:** The model *systematically under-predicts* expensive houses. At the top-right of the plot (Actual Log(Price) > 12.5), most points fall below the red $y=x$ line. The unconstrained tree overfit the training data and didn't learn the pattern for high-value homes.

3.  **RidgeRegression (alpha=1.0):**
    * **Result:** This is the best model by far (R-squared ~0.90).
    * **Why:** The error plot is a textbook example of a **good fit**. The points form a *tight, uniform cloud* around the red line. This means the errors are small and, crucially, **random**. The model does not appear to have any systematic bias; it predicts cheap and expensive homes equally well. The regularization (alpha) successfully prevented overfitting and found a stable, generalizable pattern.



# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


In [ ]:
%%timeit -n 1 -r 1
# Tuning Ridge Regression (Linear Model)

# Define the hyperparameter distribution to search
# We'll search for 'alpha' (regularization strength) in a wide range
param_dist_ridge = {
    'alpha': loguniform(1e-1, 1e2),  # Search 0.1 to 100 on a log scale
    'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga'],
    'max_iter': randint(1000, 5000)  # Tune max_iter to prevent convergence warnings
}

# Set up RandomizedSearchCV
# n_iter=50 means it will try 50 different random combinations
# cv=5 means 5-fold cross-validation
# 'neg_root_mean_squared_error' is the metric to optimize (we want to minimize RMSE)
ridge_random_search = RandomizedSearchCV(
    Ridge(),
    param_distributions=param_dist_ridge,
    n_iter=100, # Increased iterations for larger search space
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

ridge_random_search.fit(X_train_processed, ty_train)

# Print the results
print(f"Best parameters found: {ridge_random_search.best_params_}")
print(f"Best CV RMSE score: {-ridge_random_search.best_score_:.4f}")

In [ ]:
%%timeit -n 1 -r 1
# Tuning Decision Tree

# Define the hyperparameter grid to search
param_grid_tree = {
    'max_depth': [5, 7, 10, 15, None], # How deep the tree can go
    'min_samples_leaf': [5, 10, 20, 30], # Min samples required at a leaf node
    'min_samples_split': [10, 20, 40, 60], # Min samples required to split a node
    'max_features': ['sqrt', 'log2', None] # Number of features to consider at each split
}

# Set up GridSearchCV
grid_search_tree = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid=param_grid_tree,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search_tree.fit(X_train_processed, ty_train)

# Print the results
print(f"Best parameters found: {grid_search_tree.best_params_}")
print(f"Best CV RMSE score: {-grid_search_tree.best_score_:.4f}")

# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

### Non-Technical Summary (for Real Estate Executives)

We built a data-driven model that can accurately predict house sale prices in Ames, Iowa. 
Our solution identifies the key drivers of value—like overall quality, neighborhood, and living area—and uses this information to estimate prices, achieving a high degree of accuracy (over 90% R-squared) on new listings. This tool can help our agents and clients set and negotiate prices with much greater confidence.

### Technical Summary (for ML Team Lead)

We constructed a complete data preparation and modeling pipeline to predict `log(SalePrice)`. 
The solution's core is a Scikit-Learn `ColumnTransformer` that performs comprehensive preprocessing: it imputes missing values contextually (e.g., 'NA' to 'None'), applies ordinal and one-hot encoding, and scales all numeric features. 
Our final model is a `Ridge` regression, which we tuned using `RandomizedSearchCV` (best `alpha` ~16.06, `solver`='saga'), achieving a cross-validated RMSLE of approximately 0.118 and a test R-squared of over 0.90. This regularized linear model significantly outperformed an unregularized linear model (which failed due to multicollinearity) and a tuned `DecisionTree` (CV RMSLE ~0.190).

# 9. Submit your predictions to the competition

In [ ]:
# Final Model

test_data = pd.read_csv(
    "/kaggle/input/house-prices-advanced-regression-techniques/test.csv", 
    index_col="Id"
)
best_ridge = Ridge(
    alpha=16.056406613556394, 
    max_iter=2969, 
    solver='saga',
    random_state=42
)

# Transform the test data using the *already fitted* preprocessor
test_data_processed = preprocessor.transform(test_data)

# Predict using the best_ridge model (already fitted on all training data)
best_ridge.fit(X_train_processed, ty_train)
log_predictions = best_ridge.predict(test_data_processed)

In [ ]:
final_predictions = np.expm1(log_predictions)

test_predictions = pd.DataFrame({
        'Id': test_data.index,
        'SalePrice': final_predictions
    })

In [ ]:
submission_df = pd.DataFrame({
    'Id': test_data.index,
    'SalePrice': final_predictions
})

submission_df.to_csv('submission.csv', index=False)

In [ ]:
print(submission_df.head().to_string())